# Case Study 4: Factoid Question Answering

Retriever-reader pipeline over Comcast complaints with hybrid retrieval, answer-type gating and calibrated abstention. Readers are also benchmarked on SQuAD.

In [ ]:
import glob, json, os, subprocess, sys, time
from pathlib import Path
import numpy as np
import pandas as pd
import torch

PACKAGES = []
KAGGLE = os.path.isdir("/kaggle/working")
if KAGGLE:
    if PACKAGES:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PACKAGES], check=True)
    hits = glob.glob("/kaggle/input/**/telelens/__init__.py", recursive=True)
    if hits:
        ROOT = Path(hits[0]).parents[2]
    else:
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/BharatChandra-sys/NLP-case-study.git", "/tmp/repo"], check=True)
        ROOT = Path("/tmp/repo")
    RESULTS = Path("/kaggle/working/results/qa")
else:
    ROOT = Path.cwd().resolve().parent
    RESULTS = ROOT / "results" / "qa"
sys.path.insert(0, str(ROOT / "src"))
RESULTS.mkdir(parents=True, exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
print("device", DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "")


def save_json(name, payload):
    (RESULTS / name).write_text(json.dumps(payload, indent=2, default=float), encoding="utf-8")


In [ ]:
from telelens.data import load_comcast, load_gold, load_squad, load_telecom
from telelens.embeddings import log_odds_polarity, polarity_counterfit, train_sgns
from telelens.metrics import bootstrap_ci, exact_match, retrieval_scores, selective_qa, token_f1
from telelens.qa import HybridRetriever, QAPipeline, Reader, answer_type, make_passages, rule_answer
from telelens.text import bind_negation, tokenize
from telelens import viz

comcast = load_comcast()
telecom = load_telecom(10000)
gold = load_gold(ROOT / "data" / "gold" / "qa_gold.jsonl")
dev, test = [g for g in gold if g["split"] == "dev"], [g for g in gold if g["split"] == "test"]
passages = make_passages(comcast)
print(len(comcast), "documents", len(passages), "passages", len(dev), "dev", len(test), "test",
      sum(g["answerable"] for g in test), "answerable test")

## Reader benchmark on SQuAD with the gold paragraph supplied

In [ ]:
squad = load_squad()
rng = np.random.default_rng(SEED)
sample = [squad[i] for i in rng.choice(len(squad), 2000, replace=False)]
READERS = {"DistilBERT-SQuAD": "distilbert-base-cased-distilled-squad", "RoBERTa-SQuAD2": "deepset/roberta-base-squad2"}
readers = {name: Reader(model, DEVICE) for name, model in READERS.items()}
bench = []
for name, reader in readers.items():
    start = time.time()
    spans = reader.spans([s["question"] for s in sample], [s["context"] for s in sample])
    elapsed = time.time() - start
    em = [exact_match(found[0]["text"] if found else "", s["answers"]) for found, s in zip(spans, sample)]
    f1 = [token_f1(found[0]["text"] if found else "", s["answers"]) for found, s in zip(spans, sample)]
    lo, hi = bootstrap_ci(em)
    bench.append({"reader": name, "questions": len(sample), "em": np.mean(em), "em_lo": lo, "em_hi": hi, "f1": np.mean(f1), "seconds": elapsed})
bench = pd.DataFrame(bench)
bench.to_csv(RESULTS / "squad_reader_benchmark.csv", index=False)
bench.round(4)

## Embeddings trained on both telecom corpora; gold questions are never used

In [ ]:
texts = comcast["text"].tolist() + telecom["text"].tolist()
ratings = comcast["rating"].tolist()
labels = [1 if r >= 4 else 0 if r == 1 else -1 for r in ratings] + telecom["label"].tolist()
plain = [tokenize(t) for t in texts]
bound = [bind_negation(d) for d in plain]
sgns, _ = train_sgns(plain, device=DEVICE)
base, _ = train_sgns(bound, device=DEVICE, bound=True)
known = [i for i, y in enumerate(labels) if y >= 0]
pa, stats = polarity_counterfit(base, log_odds_polarity([bound[i] for i in known], [labels[i] for i in known]), device=DEVICE)
retriever = HybridRetriever(passages, {"sgns": sgns, "pa_w2v": pa})
print(stats)

## Retrieval: document rank of the evidence complaint

In [ ]:
MODES = {"BM25": "bm25", "SGNS dense": "sgns", "PA-W2V dense": "pa_w2v", "BM25 + SGNS": "bm25+sgns", "BM25 + PA-W2V": "bm25+pa_w2v",
         "BM25 rerank SGNS 0.25": "rerank:sgns:0.25", "BM25 rerank SGNS 0.5": "rerank:sgns:0.5",
         "BM25 rerank PA-W2V 0.25": "rerank:pa_w2v:0.25", "BM25 rerank PA-W2V 0.5": "rerank:pa_w2v:0.5"}
retrieval = []
for label, mode in MODES.items():
    for split, items in (("dev", dev), ("test", test)):
        ranks = [retriever.doc_rank(retriever.rank(g["question"], mode, 100), g["doc_id"]) for g in items if g["answerable"]]
        retrieval.append({"retriever": label, "split": split, **retrieval_scores(ranks)})
retrieval = pd.DataFrame(retrieval)
retrieval.to_csv(RESULTS / "retrieval.csv", index=False)
best_mode_label = retrieval[retrieval.split == "dev"].sort_values("mrr", ascending=False).iloc[0]["retriever"]
print("best on dev", best_mode_label)
retrieval.round(4)

## Tune reader, gate, retrieval weight and abstention threshold on dev only

In [ ]:
pipelines = {(label, name): QAPipeline(retriever, reader, mode=MODES[label]) for label in MODES for name, reader in readers.items()}


def evaluate(pipeline, items, gate, weight):
    results = []
    for g in items:
        best, _ = pipeline.answer(g["question"], gate=gate, retrieval_weight=weight)
        results.append({"id": g["id"], "question": g["question"], "answerable": g["answerable"], "gold": g["answers"], "prediction": best["answer"],
                        "score": best["score"], "type": best["type"], "doc_id": best["doc_id"], "gold_doc": g["doc_id"],
                        "em": exact_match(best["answer"], g["answers"]) if g["answerable"] else 0.0, "f1": token_f1(best["answer"], g["answers"]) if g["answerable"] else 0.0})
    return results


def forced(results):
    answerable = [r for r in results if r["answerable"]]
    return float(np.mean([r["em"] for r in answerable])), float(np.mean([r["f1"] for r in answerable]))


search = []
for (label, name), pipeline in pipelines.items():
    for gate in (False, True):
        for weight in (0.0, 1.0, 2.0, 4.0):
            em, f1 = forced(evaluate(pipeline, dev, gate, weight))
            search.append({"retriever": label, "reader": name, "gate": gate, "weight": weight, "dev_em": em, "dev_f1": f1})
search = pd.DataFrame(search).sort_values(["dev_em", "dev_f1"], ascending=False)
search.to_csv(RESULTS / "dev_search.csv", index=False)
chosen = search.iloc[0]
print(chosen.to_dict())
best_pipeline = pipelines[(chosen.retriever, chosen.reader)]
dev_results = evaluate(best_pipeline, dev, bool(chosen.gate), float(chosen.weight))
candidates = sorted({r["score"] for r in dev_results if r["score"] > -1e8})
sweep = pd.DataFrame([selective_qa(dev_results, tau) for tau in candidates])
tau = float(sweep.sort_values(["overall_accuracy", "answer_coverage"], ascending=False).iloc[0]["tau"])
print("tau", tau)
search.head(8).round(4)

## Held-out test: ablation, abstention and answer-type breakdown

In [ ]:
ablation = []
def add_row(label, results):
    answerable = [r for r in results if r["answerable"]]
    em, f1 = forced(results)
    lo, hi = bootstrap_ci([r["em"] for r in answerable])
    ablation.append({"system": label, "em": em, "em_lo": lo, "em_hi": hi, "f1": f1})


rule_results = []
for g in test:
    answer, score = rule_answer(g["question"], retriever, MODES[chosen.retriever])
    rule_results.append({"answerable": g["answerable"], "em": exact_match(answer, g["answers"]) if g["answerable"] else 0.0, "f1": token_f1(answer, g["answers"]) if g["answerable"] else 0.0})
add_row("Rule extractor (regex types)", rule_results)
reader_name = chosen.reader
weight = float(chosen.weight)
add_row(f"BM25 + {reader_name}, no gate", evaluate(pipelines[("BM25", reader_name)], test, False, weight))
add_row(f"BM25 + {reader_name}, gate", evaluate(pipelines[("BM25", reader_name)], test, True, weight))
add_row(f"BM25+SGNS + {reader_name}, gate", evaluate(pipelines[("BM25 + SGNS", reader_name)], test, True, weight))
add_row(f"BM25+PA-W2V + {reader_name}, gate", evaluate(pipelines[("BM25 + PA-W2V", reader_name)], test, True, weight))
add_row(f"BM25 rerank PA-W2V 0.25 + {reader_name}, gate", evaluate(pipelines[("BM25 rerank PA-W2V 0.25", reader_name)], test, True, weight))
other = [n for n in READERS if n != reader_name][0]
add_row(f"Dev-selected retriever + {other}, gate", evaluate(pipelines[(chosen.retriever, other)], test, True, weight))
test_results = evaluate(best_pipeline, test, bool(chosen.gate), weight)
add_row("Dev-selected system", test_results)
ablation = pd.DataFrame(ablation)
ablation.to_csv(RESULTS / "test_ablation.csv", index=False)
selective = selective_qa(test_results, tau)
no_abstain = selective_qa(test_results, -1e8)
save_json("test_selective.json", {"chosen": {k: (v.item() if hasattr(v, "item") else v) for k, v in chosen.to_dict().items()}, "tau": tau, "with_abstention": selective, "no_abstention": no_abstain})
pd.DataFrame(test_results).to_json(RESULTS / "test_predictions.jsonl", orient="records", lines=True)
answerable = pd.DataFrame([r for r in test_results if r["answerable"]])
by_type = answerable.groupby("type").agg(questions=("em", "size"), em=("em", "mean"), f1=("f1", "mean")).reset_index()
by_type.to_csv(RESULTS / "test_by_type.csv", index=False)
curve = [selective_qa(test_results, t) for t in sorted({r["score"] for r in test_results if r["score"] > -1e8})]
viz.lines(RESULTS / "qa_abstention.png", [("Answered EM", [c["answer_coverage"] for c in curve], [c["answered_em"] for c in curve]),
                                           ("Unanswerable rejected", [c["answer_coverage"] for c in curve], [c["unanswerable_rejection"] for c in curve])],
          "Abstention trade-off on the held-out questions", "Share of answerable questions answered", "Rate", ylim=(0, 1.05))
ordered = ablation.sort_values("em")
viz.hbar(RESULTS / "qa_ablation.png", ordered["system"].tolist()[::-1], ordered["em"].tolist()[::-1], "Exact match on held-out answerable questions", "Exact match")
display(ablation.round(4), selective, no_abstain, by_type.round(4))

In [ ]:
errors = answerable[answerable.em == 0][["question", "gold", "prediction", "f1", "doc_id", "gold_doc"]]
errors.to_csv(RESULTS / "test_errors.csv", index=False)
errors.head(15)

## Recorded Kaggle results

Executed on a Kaggle T4 GPU as `telelens-04-factoid-qa`. Files are in `results/qa/`.

**squad_reader_benchmark.csv**

| reader | questions | em | em_lo | em_hi | f1 | seconds |
|---|---|---|---|---|---|---|
| DistilBERT-SQuAD | 2000 | 0.785 | 0.7675 | 0.8025 | 0.8599 | 25.9437 |
| RoBERTa-SQuAD2 | 2000 | 0.844 | 0.8285 | 0.8595 | 0.9074 | 44.5896 |

**retrieval.csv**

| retriever | split | recall@1 | recall@5 | recall@20 | mrr |
|---|---|---|---|---|---|
| BM25 | dev | 0.7059 | 0.8235 | 0.8824 | 0.7632 |
| BM25 | test | 0.6029 | 0.8235 | 0.8676 | 0.6993 |
| SGNS dense | dev | 0.1765 | 0.2941 | 0.3824 | 0.2236 |
| SGNS dense | test | 0.1618 | 0.3235 | 0.5 | 0.2344 |
| PA-W2V dense | dev | 0.1176 | 0.1765 | 0.3824 | 0.1652 |
| PA-W2V dense | test | 0.1324 | 0.2647 | 0.3971 | 0.1919 |
| BM25 + SGNS | dev | 0.3235 | 0.4706 | 0.8235 | 0.4106 |
| BM25 + SGNS | test | 0.25 | 0.6176 | 0.8529 | 0.4274 |
| BM25 + PA-W2V | dev | 0.2353 | 0.5 | 0.8235 | 0.3681 |
| BM25 + PA-W2V | test | 0.2353 | 0.5588 | 0.8529 | 0.3913 |
| BM25 rerank SGNS 0.25 | dev | 0.6471 | 0.7941 | 0.8824 | 0.7291 |
| BM25 rerank SGNS 0.25 | test | 0.6029 | 0.8235 | 0.8676 | 0.7033 |
| BM25 rerank SGNS 0.5 | dev | 0.6471 | 0.7941 | 0.8824 | 0.717 |
| BM25 rerank SGNS 0.5 | test | 0.6029 | 0.8235 | 0.8676 | 0.6998 |
| BM25 rerank PA-W2V 0.25 | dev | 0.6471 | 0.8235 | 0.8824 | 0.7297 |
| BM25 rerank PA-W2V 0.25 | test | 0.6029 | 0.8382 | 0.8824 | 0.6969 |
| BM25 rerank PA-W2V 0.5 | dev | 0.6176 | 0.7941 | 0.8824 | 0.705 |
| BM25 rerank PA-W2V 0.5 | test | 0.5882 | 0.8382 | 0.8824 | 0.6954 |

**test_ablation.csv**

| system | em | em_lo | em_hi | f1 |
|---|---|---|---|---|
| Rule extractor (regex types) | 0.3235 | 0.2206 | 0.4412 | 0.3382 |
| BM25 + RoBERTa-SQuAD2, no gate | 0.5882 | 0.4706 | 0.7059 | 0.6324 |
| BM25 + RoBERTa-SQuAD2, gate | 0.6176 | 0.5 | 0.7353 | 0.6422 |
| BM25+SGNS + RoBERTa-SQuAD2, gate | 0.5441 | 0.4265 | 0.6618 | 0.5613 |
| BM25+PA-W2V + RoBERTa-SQuAD2, gate | 0.5 | 0.3824 | 0.6176 | 0.5319 |
| BM25 rerank PA-W2V 0.25 + RoBERTa-SQuAD2, gate | 0.6324 | 0.5147 | 0.7353 | 0.6495 |
| Dev-selected retriever + DistilBERT-SQuAD, gate | 0.5735 | 0.4559 | 0.6768 | 0.5956 |
| Dev-selected system | 0.6176 | 0.5 | 0.7353 | 0.6422 |

**test_by_type.csv**

| type | questions | em | f1 |
|---|---|---|---|
| COUNT | 1 | 1.0 | 1.0 |
| DATE | 2 | 0.5 | 0.5 |
| DURATION | 13 | 0.6154 | 0.6923 |
| ENTITY | 24 | 0.625 | 0.625 |
| MONEY | 26 | 0.5769 | 0.6026 |
| TIME | 2 | 1.0 | 1.0 |

![qa_ablation.png](../results/qa/qa_ablation.png)

![qa_abstention.png](../results/qa/qa_abstention.png)

